# CSP3 maize GPP on non-irrigated corn surfaces

Google Colab notebook (Earth Engine Python API + **geemap** + **rasterio**).

This notebook:

1. Mounts Drive and reads `PHD/CSP3` workbooks plus `PHD/non_irrigated_corn/clean` GeoTIFFs.
2. Extracts 2001–2017 **ops** growing periods (first PLANT → last HARVEST, ±14 days) from `2001-2017 site-history CSP3.xlsx` sheet **CSP3**.
3. Extracts maize **phenology**, All-IMZ `MC_AGB`, and HI from annual Site 3 workbooks (A1 must say Maize).
4. Builds seasonal Landsat GPP rasters on each year’s non-irrigated corn mask (EPSG:5070, 30 m). CSP3 / US-Ne3 is a **calibration point**, not the AOI.
5. Writes products to `PHD/CSP3_GPP_outputs` without overwriting source workbooks or AOI GeoTIFFs.

**Not done here:** converting GPP to biomass, multiplying HI into GPP, clipping to the 1793 field polygon, reusing 2017 as a 2019/2021 mask, or labeling R5 as R6.


## 1. Install, mount Drive, authenticate Earth Engine

In [ ]:
# Colab: install and authenticate
!pip -q install earthengine-api geemap rasterio openpyxl pandas geopandas

from google.colab import drive, auth
drive.mount("/content/drive")

import os, sys, shutil
from pathlib import Path

import ee, geemap, rasterio
import pandas as pd
import numpy as np

# Set your Earth Engine Cloud project id (required for ee.Initialize).
EE_PROJECT = os.environ.get("EE_PROJECT", "your-ee-project-id")

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
Map = geemap.Map()
Map


In [ ]:
# Paths: Colab Drive layout. Helpers live beside this notebook in the repo
# and are also copied to PHD/CSP3_GPP_outputs.

PHD = Path("/content/drive/MyDrive/PHD")
CSP3 = PHD / "CSP3"
AOI_CLEAN = PHD / "non_irrigated_corn" / "clean"
AOI_PARENT = PHD / "non_irrigated_corn"
OUT = PHD / "CSP3_GPP_outputs"
RASTERS = OUT / "rasters"
OUT.mkdir(parents=True, exist_ok=True)
RASTERS.mkdir(parents=True, exist_ok=True)

# Import helpers from the folder that contains this notebook (Drive copy or repo clone).
HERE = Path.cwd()
for candidate in [HERE, OUT, Path("/content"), Path("/workspace/csp3_maize_gpp")]:
    if (candidate / "config.py").exists():
        sys.path.insert(0, str(candidate))
        HELPERS = candidate
        break
else:
    raise FileNotFoundError(
        "config.py not found. Upload the csp3_maize_gpp helper .py files "
        "next to this notebook or into PHD/CSP3_GPP_outputs."
    )

import config
config.EE_PROJECT = EE_PROJECT
print("helpers:", HELPERS)
print("PHD exists:", PHD.exists(), "CSP3 exists:", CSP3.exists(), "AOI clean exists:", AOI_CLEAN.exists())


## 2. Discover workbooks and crop-validate A1

In [ ]:
from discover_workbooks import discover_local_workbooks, preferred_annual
from extract_site_history import extract_site_history
from extract_parameters import extract_all_maize
from build_outputs import build_outputs
from config import SITE_HISTORY_FILE, AOI_YEARS, US_NE3_LAT, US_NE3_LON

extra = []
biomass = CSP3 / "Biomass Sampling_2003_2021"
if biomass.exists():
    extra.append(biomass)

discovery = discover_local_workbooks(CSP3, extra_dirs=extra)
disc_df = pd.DataFrame(discovery)
display(disc_df[["year", "role", "filename", "a1_crop", "a1_raw", "preferred", "top_level"]])

annual = preferred_annual(discovery)
print("Preferred annual workbooks:", sorted(annual))
for year, rec in sorted(annual.items()):
    print(f"  {year}: A1={rec.get('a1_raw')!r} crop={rec.get('a1_crop')}")


## 3. Site-history ops dates (2001–2017) and maize phenology / MC_AGB / HI

In [ ]:
history = CSP3 / SITE_HISTORY_FILE["name"]
ops = extract_site_history(history)
ops_df = pd.DataFrame(ops)
display(ops_df[["year", "crop", "crop_source", "ops_plant_date", "ops_harvest_date", "buffered_start", "buffered_end"]])

phenology = extract_all_maize(annual)
pheno_df = pd.DataFrame(phenology)
cols = [c for c in [
    "year", "planting_doy", "planting_date", "end_stage", "end_is_provisional",
    "sampling_date", "buffered_start", "buffered_end", "mc_agb", "hi", "hi_method", "hi_status"
] if c in pheno_df.columns]
display(pheno_df[cols] if len(pheno_df) else pheno_df)

# Hard-check: never substitute harvest for R6 / provisional R5
assert pheno_df.set_index("year").loc[2013, "sampling_date"] == "2013-09-25"
assert ops_df.set_index("year").loc[2013, "ops_harvest_date"] == "2013-10-22"
assert pheno_df.set_index("year").loc[2017, "end_stage"] == "R5"
assert pheno_df.set_index("year").loc[2017, "end_is_provisional"] == True
print("Phenology vs harvest dates are distinct. Checkpoints passed.")


## 4. Write audit tables to `PHD/CSP3_GPP_outputs`

In [ ]:
result = build_outputs(output_dir=OUT, workbooks_dir=CSP3)
for name, path in result["files"].items():
    print(name, "->", path)
print("gee_ok", result.get("gee_ok"), "n_ops", result.get("n_ops"), "n_phenology", result.get("n_phenology"))


## 5. Seasonal GPP rasters on yearly non-irrigated corn surfaces

For each maize year with a `*_clean.tif` mask (2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017):

- `UMT/NTSG/v2/LANDSAT/GPP` filtered to the **ops** buffered window (`filterDate` end exclusive)
- QC 10 or 11 only, GPP × 0.0001
- Sum to `GPP_kgCm2_season`; count to `n_obs`
- Reproject to the mask grid (EPSG:5070, 30 m) and mask with rasterio (`value > 0`)
- Do **not** apply HI or `MC_AGB`
- Soybean years: no GeoTIFF. 2019/2021: `aoi_raster_missing` (do not reuse 2017)


In [ ]:
from datetime import date
from aoi import find_aoi_raster, read_aoi_profile, assert_aoi_grid
from gee_gpp import export_year_rasters, seasonal_gpp_images, add_map_layers, zonal_from_raster

zonal_path = OUT / "csp3_gpp_zonal_nonirr_corn.csv"
zonal = pd.read_csv(zonal_path)
export_logs = []

for _, row in zonal.iterrows():
    year = int(row["year"])
    crop = row.get("crop")
    if crop != "maize" or year not in AOI_YEARS:
        if crop != "maize":
            soy_sum = RASTERS / f"gpp_seasonal_sum_nonirr_corn_{year}.tif"
            assert not soy_sum.exists(), f"soybean year {year} must not have a GPP raster"
        continue
    if pd.isna(row.get("buffered_start")):
        export_logs.append({"year": year, "status": "ops_dates_missing"})
        continue
    aoi = find_aoi_raster(year, AOI_CLEAN, AOI_PARENT)
    print(year, "AOI", aoi.get("status"), aoi.get("path"))
    if not aoi.get("path"):
        export_logs.append({"year": year, "status": aoi.get("status")})
        continue
    profile = read_aoi_profile(Path(aoi["path"]))
    assert_aoi_grid(profile, Path(aoi["path"]))
    info = export_year_rasters(
        year,
        date.fromisoformat(str(row["buffered_start"])[:10]),
        date.fromisoformat(str(row["buffered_end"])[:10]),
        date.fromisoformat(str(row["gee_filter_end_exclusive"])[:10]),
        Path(aoi["path"]),
        RASTERS,
    )
    export_logs.append(info)
    print(year, info.get("status"), info.get("exported_via"), info.get("gpp_mean"))

pd.DataFrame(export_logs)


## 6. rasterio QC and zonal CSV (after exports exist)

In [ ]:
rows = []
for _, row in zonal.iterrows():
    year = int(row["year"])
    rec = dict(row)
    sum_p = RASTERS / f"gpp_seasonal_sum_nonirr_corn_{year}.tif"
    nobs_p = RASTERS / f"gpp_nobs_nonirr_corn_{year}.tif"
    aoi = find_aoi_raster(year, AOI_CLEAN, AOI_PARENT)
    if rec.get("crop") != "maize":
        rec["gpp_raster_status"] = "skipped_not_maize"
        assert not sum_p.exists(), f"unexpected GPP raster for {year}"
        rows.append(rec)
        continue
    if year not in AOI_YEARS:
        rec["gpp_raster_status"] = "aoi_raster_missing"
        assert not sum_p.exists(), "do not invent 2019/2021 masks"
        rows.append(rec)
        continue
    if not sum_p.exists() or not nobs_p.exists() or not aoi.get("path"):
        rec["gpp_raster_status"] = rec.get("gpp_raster_status") or "pending_export"
        rows.append(rec)
        continue
    stats = zonal_from_raster(sum_p, nobs_p, Path(aoi["path"]))
    rec.update({
        "gpp_mean": stats["gpp_mean"],
        "gpp_median": stats["gpp_median"],
        "n_corn_pixels": stats["n_corn_pixels"],
        "gpp_raster_status": "ok",
        "hi_applied_to_gpp": False,
        "mc_agb_applied_to_gpp": False,
    })
    with rasterio.open(sum_p) as ds:
        tags = " ".join(str(x) for x in ds.descriptions)
        assert "HI" not in tags.upper()
        assert "MC_AGB" not in tags.upper()
    rows.append(rec)

zonal_out = pd.DataFrame(rows)
zonal_out.to_csv(OUT / "csp3_gpp_zonal_nonirr_corn.csv", index=False)
display(zonal_out[["year", "crop", "gpp_raster_status", "gpp_mean", "gpp_median", "n_corn_pixels"]])


## 7. geemap preview (one maize year + US-Ne3)

In [ ]:
PREVIEW_YEAR = 2013
aoi = find_aoi_raster(PREVIEW_YEAR, AOI_CLEAN, AOI_PARENT)
z = zonal.set_index("year").loc[PREVIEW_YEAR]
seasonal, n_obs, _ = seasonal_gpp_images(
    date.fromisoformat(str(z["buffered_start"])[:10]),
    date.fromisoformat(str(z["gee_filter_end_exclusive"])[:10]),
)
preview = add_map_layers(PREVIEW_YEAR, seasonal, Path(aoi["path"]) if aoi.get("path") else None)
preview


## 8. Copy helpers next to Drive outputs

In [ ]:
for fname in [
    "config.py", "headers.py", "formulas.py", "discover_workbooks.py",
    "extract_site_history.py", "extract_parameters.py", "aoi.py", "gee_gpp.py",
    "build_outputs.py", "run_pipeline.py", "CSP3_GPP_Colab.ipynb",
]:
    src = HELPERS / fname
    if src.exists():
        shutil.copy2(src, OUT / fname)
        print("copied", fname)
print("Done. Outputs:", list(p.name for p in sorted(OUT.iterdir()) if p.is_file())[:20])
